# Tabular ML 실습 노트북

이 노트북은 학부생 강의 "Tabular ML"의 실습용 자료입니다.
대표적인 표형식 데이터셋을 활용해 다음 6가지 방법론을 한 번씩 실험해 봅니다.

1. **데이터셋 다운로드** — UCI MAGIC Gamma Telescope (binary classification, 비선형성이 강한 물리 데이터)
2. **Logistic Regression** — 가장 단순한 선형 모델
3. **Decision Tree / XGBoost** — 전통적 비선형 모델, 부스팅 트리
4. **Deep Architecture (TabM)** — MLP 기반 최신 딥러닝 모델
5. **LLM 실험** — 행(row)을 자연어로 직렬화한 뒤 작은 오픈 LLM으로 분류
6. **TabPFN** — Tabular Foundation Model

> **권장 런타임**: Colab GPU (T4) — TabM, LLM, TabPFN 단계에서 GPU 사용 시 훨씬 빠릅니다.
> **메뉴**: `런타임` → `런타임 유형 변경` → `T4 GPU`


## 0. 환경 설정

필요한 패키지를 설치합니다. Colab에는 numpy/pandas/sklearn/torch가 이미 설치되어 있으므로,
추가로 필요한 `xgboost`, `tabpfn`, `transformers`, `accelerate`만 설치합니다.


In [ ]:
# 패키지 설치 (Colab 기준)
# 처음 한 번만 실행하면 됩니다. 설치 후 런타임 재시작이 필요할 수도 있습니다.
!pip install -q \
    "xgboost==2.1.3" \
    "tabpfn==2.0.9" \
    "transformers>=4.45,<5.0" \
    "accelerate>=0.34"


In [ ]:
# 공통 import
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

# 재현성을 위한 시드 고정
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# GPU 사용 가능 여부 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)


## 1. 데이터셋: MAGIC Gamma Telescope

**Task**: 첸코프(MAGIC) 망원경이 관측한 샤워 이미지에서 추출한 10개의
형상/대칭/모멘트 feature를 보고, 그 이벤트가 **감마선 신호(gamma)** 인지
**우주선 하드론 배경(hadron)** 인지 판별하는 binary classification.

- 19,020개 샘플 — gamma 12,332 / hadron 6,688 (대략 65% / 35%)
- Feature 10개 모두 **수치형(continuous)**. 결측치 없음.
- 결정경계가 **강하게 비선형**이라 선형 모델과 비선형 모델의 차이가 잘 드러남
  → 강의에서 "왜 트리/딥러닝이 필요한가" 보여주기에 적합.
- 출처: <https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope>

> *Adult Income 같은 인구통계 데이터는 선형성이 강해 LR이 종종 비등하게 나옵니다.*
> *MAGIC은 물리적 비선형성이 본질적이라 모델 간 차이가 보다 명확히 보입니다.*


In [ ]:
# 1) 데이터 다운로드 — UCI 서버에서 단일 CSV 파일을 직접 받아옵니다.
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/magic/magic04.data"

NUM_COLS = [
    "fLength",   # 주축 길이
    "fWidth",    # 부축 너비
    "fSize",     # 픽셀 합의 log
    "fConc",     # 상위 2 픽셀 / fSize
    "fConc1",    # 최상위 픽셀 / fSize
    "fAsym",     # 주축 비대칭도
    "fM3Long",   # 주축 3차 모멘트
    "fM3Trans",  # 부축 3차 모멘트
    "fAlpha",    # 주축과 원점 벡터 사이 각도
    "fDist",     # 원점에서 중심까지 거리
]
TARGET = "class"
COLUMNS = NUM_COLS + [TARGET]

df = pd.read_csv(DATA_URL, header=None, names=COLUMNS)
print("shape:", df.shape)
df.head()


In [ ]:
# 2) 학습/테스트 분리 (8:2, 클래스 비율 유지)
from sklearn.model_selection import train_test_split

CAT_COLS = []   # MAGIC에는 범주형 feature가 없음 — 변수만 정의해 두기

# 라벨: 'g' (gamma, 신호) → 1, 'h' (hadron, 배경) → 0
df["y"] = (df[TARGET] == "g").astype(int)

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["y"], random_state=SEED,
)
print("train:", train_df.shape, "test:", test_df.shape)
print("\n== 타겟 분포 (train) ==")
print(train_df[TARGET].value_counts(normalize=True).round(3))


### 1-1. 전처리

MAGIC은 모두 수치형이라 전처리가 매우 단순합니다.

1. **타겟 인코딩** — `g` (gamma) → 1, `h` (hadron) → 0
2. **선형/딥러닝 모델용** — StandardScaler 적용
3. **트리 모델용** — 원본 그대로 (트리는 스케일 불필요)


In [ ]:
from sklearn.preprocessing import StandardScaler

def split_xy(d):
    return d[NUM_COLS].to_numpy(dtype=float), d["y"].to_numpy()

X_train_raw, y_train = split_xy(train_df)
X_test_raw,  y_test  = split_xy(test_df)

# (A) 트리 모델용: 원본 그대로
X_train_tree = X_train_raw
X_test_tree  = X_test_raw

# (B) 선형/딥러닝 모델용: StandardScaler
scaler = StandardScaler().fit(X_train_raw)
X_train_lin = scaler.transform(X_train_raw)
X_test_lin  = scaler.transform(X_test_raw)

print("Tree 형태  :", X_train_tree.shape, X_test_tree.shape)
print("Linear 형태:", X_train_lin.shape,  X_test_lin.shape)


In [ ]:
# 평가 지표를 한 곳에서 계산하는 헬퍼
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

# 결과를 누적할 딕셔너리
results = {}

def evaluate(name, y_true, y_pred, y_score=None):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_score) if y_score is not None else float("nan")
    results[name] = {"Accuracy": acc, "F1": f1, "AUC": auc}
    print(f"[{name}] Accuracy={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}")


## 2. Logistic Regression

가장 기본적인 baseline. **선형 결정경계**만 학습할 수 있지만, 표형식 데이터에서는
의외로 강력한 출발점입니다. one-hot encoding + standard scaling을 거친 입력을 사용합니다.


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    C=1.0,           # 규제 강도의 역수
    solver="lbfgs",
    random_state=SEED,
)
lr_model.fit(X_train_lin, y_train)

y_pred_lr  = lr_model.predict(X_test_lin)
y_score_lr = lr_model.predict_proba(X_test_lin)[:, 1]
evaluate("LogisticRegression", y_test, y_pred_lr, y_score_lr)


## 3. Decision Tree

비선형 결정경계를 만들 수 있는 가장 단순한 모델. `max_depth`로 모델 복잡도를 조절합니다.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    max_depth=8,           # 너무 깊으면 과적합
    min_samples_leaf=20,
    random_state=SEED,
)
dt_model.fit(X_train_tree, y_train)

y_pred_dt  = dt_model.predict(X_test_tree)
y_score_dt = dt_model.predict_proba(X_test_tree)[:, 1]
evaluate("DecisionTree", y_test, y_pred_dt, y_score_dt)


## 4. XGBoost

**Gradient Boosted Trees**의 대표적 구현체. 표형식 데이터에서 오랫동안 사실상 SOTA였고,
여전히 강력한 baseline입니다. 학습 중 validation 성능을 모니터링해 **early stopping**을 사용합니다.


In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# early stopping을 위해 train의 일부를 validation으로 분리
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_tree, y_train, test_size=0.2, random_state=SEED, stratify=y_train
)

xgb_model = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    eval_metric="auc",
    early_stopping_rounds=50,
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
print("최적 반복수 :", xgb_model.best_iteration)

y_pred_xgb  = xgb_model.predict(X_test_tree)
y_score_xgb = xgb_model.predict_proba(X_test_tree)[:, 1]
evaluate("XGBoost", y_test, y_pred_xgb, y_score_xgb)


## 5. Deep Architecture — TabM (간소화 버전)

**TabM** (Gorishniy et al., ICLR 2025) 은 **여러 개의 MLP를 효율적으로 앙상블**하는
최신 tabular 딥러닝 모델입니다. 핵심 아이디어는:

- *k*개의 MLP를 따로 학습하는 대신, **공유되는 weight matrix**에 각 멤버별
  **저랭크(scale) 파라미터**(α, β)를 곱해 사실상 *k*개의 다른 모델을 동시에 학습.
- 메모리/계산 비용은 한 MLP에 가깝지만 앙상블 효과를 얻을 수 있음.

여기서는 강의용으로 **간소화한 구현**을 PyTorch로 직접 작성합니다.
공식 구현은 [yandex-research/tabm](https://github.com/yandex-research/tabm) 에서 확인할 수 있습니다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# 입력 차원, 텐서 변환
INPUT_DIM = X_train_lin.shape[1]
X_tr_t = torch.tensor(np.asarray(X_train_lin), dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.long)
X_te_t = torch.tensor(np.asarray(X_test_lin),  dtype=torch.float32)
y_te_t = torch.tensor(y_test,  dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=512, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_te_t, y_te_t), batch_size=1024, shuffle=False)

print("INPUT_DIM =", INPUT_DIM)


In [ ]:
class BatchEnsembleLinear(nn.Module):
    """TabM의 핵심 모듈.

    공유 weight W (out, in) 하나에 대해,
    각 멤버 k마다 입력 측 스케일 r_k와 출력 측 스케일 s_k를 곱해
    실질적으로 W_k = diag(s_k) @ W @ diag(r_k) 형태의 서로 다른 weight를 흉내냅니다.

    입력 x: (K, B, in)
    출력 y: (K, B, out)
    """
    def __init__(self, in_features, out_features, k):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.k = k
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(k, out_features))
        # 멤버별 scale (r, s)
        self.r = nn.Parameter(torch.randn(k, in_features) * 0.5 + 1.0)
        self.s = nn.Parameter(torch.randn(k, out_features) * 0.5 + 1.0)
        nn.init.kaiming_uniform_(self.weight, a=5 ** 0.5)

    def forward(self, x):
        # x: (K, B, in) — 입력에 r 곱하기
        x = x * self.r.unsqueeze(1)
        # (K, B, in) @ (in, out) -> (K, B, out)
        x = x @ self.weight.t()
        # 출력에 s, bias 적용
        x = x * self.s.unsqueeze(1) + self.bias.unsqueeze(1)
        return x


class TabM(nn.Module):
    """간소화된 TabM.

    - K개의 멤버(MLP)를 BatchEnsembleLinear로 동시에 학습
    - 최종 예측은 K개의 logits를 평균
    """
    def __init__(self, in_dim, hidden=128, depth=3, num_classes=2, k=8, dropout=0.1):
        super().__init__()
        self.k = k
        dims = [in_dim] + [hidden] * depth
        self.layers = nn.ModuleList([
            BatchEnsembleLinear(dims[i], dims[i+1], k) for i in range(depth)
        ])
        self.head = BatchEnsembleLinear(hidden, num_classes, k)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (B, in) -> (K, B, in)
        x = x.unsqueeze(0).expand(self.k, -1, -1)
        for layer in self.layers:
            x = layer(x)
            x = F.relu(x)
            x = self.dropout(x)
        logits = self.head(x)         # (K, B, C)
        return logits

    def predict_logits(self, x):
        """K개 멤버의 평균 logits."""
        logits = self.forward(x)
        return logits.mean(dim=0)


In [ ]:
# 학습 루프
tabm = TabM(in_dim=INPUT_DIM, hidden=192, depth=3, num_classes=2, k=8, dropout=0.1).to(device)
optimizer = torch.optim.AdamW(tabm.parameters(), lr=3e-4, weight_decay=1e-5)
ce_loss = nn.CrossEntropyLoss()

EPOCHS = 20
for epoch in range(1, EPOCHS + 1):
    tabm.train()
    total, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = tabm(xb)                       # (K, B, C)
        # 각 멤버를 동일 라벨에 대해 학습 → K번 평균 loss
        loss = ce_loss(logits.reshape(-1, 2), yb.repeat(tabm.k))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * xb.size(0)
        n     += xb.size(0)
    if epoch % 4 == 0 or epoch == 1:
        print(f"epoch {epoch:02d}  train_loss={total/n:.4f}")


In [ ]:
# 평가
tabm.eval()
all_logits, all_y = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = tabm.predict_logits(xb)
        all_logits.append(logits.cpu())
        all_y.append(yb)
logits = torch.cat(all_logits)
probs  = torch.softmax(logits, dim=-1)[:, 1].numpy()
preds  = logits.argmax(dim=-1).numpy()
evaluate("TabM", y_test, preds, probs)


## 6. LLM 실험 — 작은 오픈 모델로 분류

LLM을 표형식 데이터에 적용하는 가장 단순한 방법은:

1. **직렬화 (serialization)** — 각 행을 자연어 문장으로 변환
   예) `major axis length (fLength)=28.797, minor axis width (fWidth)=16.002, ... → gamma or hadron?`
2. **프롬프트** — 라벨이 있는 예시 몇 개(few-shot)와 함께 입력
3. **출력 디코딩** — 모델이 생성한 `gamma` / `hadron` 토큰을 파싱

여기서는 **Qwen2.5-0.5B-Instruct** (작고 라이선스 친화적) 를 사용합니다.
큰 LLM이 아니므로 정확도는 baseline 수준이며, *방법론* 자체를 체험하는 것이 목적입니다.
특히 MAGIC은 feature가 추상적 물리량이라 작은 LLM이 도메인 지식을 활용하기 어렵습니다 —
모델 크기를 키운 결과는 Section 9에서 비교합니다.

⚠️ 추론이 상대적으로 느리므로 **테스트셋 중 200개만 샘플링**해서 평가합니다.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
).to(device)
llm.eval()
print("모델 로드 완료:", MODEL_ID)


In [ ]:
# 1) 행 → 자연어 직렬화 함수
# 읽기 쉽게 각 feature에 짧은 한 줄 설명을 곁들입니다.
FEATURE_DESC = {
    "fLength":  "major axis length",
    "fWidth":   "minor axis width",
    "fSize":    "log of pixel sum",
    "fConc":    "top-2 pixel ratio",
    "fConc1":   "top-1 pixel ratio",
    "fAsym":    "asymmetry along major axis",
    "fM3Long":  "3rd moment along major axis",
    "fM3Trans": "3rd moment along minor axis",
    "fAlpha":   "angle of major axis vs origin",
    "fDist":    "distance from origin",
}

def serialize_row(row: pd.Series) -> str:
    parts = [f"{FEATURE_DESC[c]} ({c})={row[c]:.3f}" for c in NUM_COLS]
    return ", ".join(parts)

# 예시 출력
print(serialize_row(train_df.iloc[0]))


In [ ]:
# 2) few-shot 예시 구성 — 양/음 클래스에서 각각 4개씩
rng = np.random.default_rng(SEED)
pos_pool = train_df.index[train_df[TARGET] == "g"]
neg_pool = train_df.index[train_df[TARGET] == "h"]
pos_idx = rng.choice(pos_pool, 4, replace=False)
neg_idx = rng.choice(neg_pool, 4, replace=False)
fewshot_idx = np.concatenate([pos_idx, neg_idx])
rng.shuffle(fewshot_idx)

INSTRUCTION = (
    "You are a classifier for the MAGIC Cherenkov telescope. "
    "Given numerical shower-image features (geometry, asymmetry, moments, angle), "
    "predict whether the event is a gamma-ray signal or a hadron background. "
    "Answer with exactly one word: gamma or hadron."
)

LABEL_TEXT = {"g": "gamma", "h": "hadron"}

def build_prompt(test_row: pd.Series) -> str:
    msgs = [{"role": "system", "content": INSTRUCTION}]
    for i in fewshot_idx:
        ex = train_df.loc[i]
        msgs.append({"role": "user",      "content": serialize_row(ex)})
        msgs.append({"role": "assistant", "content": LABEL_TEXT[ex[TARGET]]})
    msgs.append({"role": "user", "content": serialize_row(test_row)})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print(build_prompt(test_df.iloc[0])[:800], "...")


In [ ]:
# 3) 200개 샘플에 대해서만 분류
N_LLM_SAMPLES = 200
sample_idx = rng.choice(len(test_df), N_LLM_SAMPLES, replace=False)

# 두 라벨 토큰 id (앞 공백 포함/미포함 모두 고려, 대소문자 변형도)
def first_token_id(text):
    return tok(text, add_special_tokens=False).input_ids[0]

POS_IDS = list({first_token_id(t) for t in ["gamma", " gamma", "Gamma", " Gamma"]})
NEG_IDS = list({first_token_id(t) for t in ["hadron", " hadron", "Hadron", " Hadron"]})

llm_preds, llm_scores = [], []
with torch.no_grad():
    for j, idx in enumerate(sample_idx):
        row = test_df.iloc[idx]
        prompt = build_prompt(row)
        inputs = tok(prompt, return_tensors="pt").to(device)
        # 다음 토큰 logits만 계산
        out = llm(**inputs)
        next_logits = out.logits[0, -1]
        # 두 후보의 log-prob을 비교
        pos_score = torch.logsumexp(next_logits[POS_IDS], dim=0).item()
        neg_score = torch.logsumexp(next_logits[NEG_IDS], dim=0).item()
        llm_preds.append(int(pos_score > neg_score))
        # AUC 계산을 위한 score (logit margin)
        llm_scores.append(pos_score - neg_score)
        if (j + 1) % 50 == 0:
            print(f"  진행: {j+1}/{N_LLM_SAMPLES}")

y_sub = y_test[sample_idx]
evaluate("LLM (Qwen2.5-0.5B, 200 samples)", y_sub,
         np.array(llm_preds), np.array(llm_scores))


## 7. TabPFN — Tabular Foundation Model

**TabPFN v2** (Hollmann et al., Nature 2025) 은 사전학습된 transformer 하나가
**fitting 없이** 새 데이터셋을 *in-context*로 분류/회귀합니다.

특징:
- 학습 과정이 사실상 없음 (`fit`은 데이터를 메모리에 올리는 정도)
- 작은~중간 규모 데이터셋(≤10K samples, ≤500 features)에서 매우 강력
- 추론 시 GPU가 있으면 훨씬 빠름

MAGIC 학습셋은 약 15,200개로 TabPFN 권장 범위(≤10K)를 살짝 넘으므로
**10,000개로 subsample**해 사용합니다.


In [ ]:
from tabpfn import TabPFNClassifier

# TabPFN의 권장 범위에 맞춰 학습셋 subsample
N_TABPFN = 10_000
sub_idx = rng.choice(len(X_train_tree), N_TABPFN, replace=False)

X_tr_pfn = X_train_tree[sub_idx]
y_tr_pfn = y_train[sub_idx]

tabpfn = TabPFNClassifier(device=device.type, ignore_pretraining_limits=False)
tabpfn.fit(X_tr_pfn, y_tr_pfn)

y_pred_pfn  = tabpfn.predict(X_test_tree)
y_score_pfn = tabpfn.predict_proba(X_test_tree)[:, 1]
evaluate(f"TabPFN ({N_TABPFN} train)", y_test, y_pred_pfn, y_score_pfn)


## 8. 결과 비교

지금까지 실험한 모든 모델의 정확도/F1/AUC를 한 번에 봅니다.


In [ ]:
summary = pd.DataFrame(results).T.round(4)
summary = summary.sort_values("AUC", ascending=False)
summary


## 9. (추가 실험) LLM 성능 끌어올리기 — 더 큰 모델로 swap

방금 본 LLM 결과가 다른 모델보다 한참 낮은 이유는 **모델이 너무 작기 때문**입니다 (0.5B).
직렬화 방식이나 프롬프트는 그대로 두고, **모델만 키워도** 정확도가 올라갈 수 있습니다.

여기서는 **Qwen2.5-3B-Instruct** 로 바꿔 봅니다.
- fp16 으로 약 6GB → Colab T4 (16GB) 에서 사용 가능
- few-shot 예시, 직렬화, 평가 코드는 그대로 재사용

In [ ]:
# 메모리 정리 후 더 큰 모델 로드
del llm, tok
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

BIG_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

tok = AutoTokenizer.from_pretrained(BIG_MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(
    BIG_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
llm.eval()
print("모델 로드 완료:", BIG_MODEL_ID)


In [ ]:
# 같은 평가 루프를 더 큰 모델로 그대로 다시 실행
# (POS_IDS/NEG_IDS도 새 tokenizer로 다시 계산)
POS_IDS = list({first_token_id(t) for t in ["gamma", " gamma", "Gamma", " Gamma"]})
NEG_IDS = list({first_token_id(t) for t in ["hadron", " hadron", "Hadron", " Hadron"]})

llm_preds_big, llm_scores_big = [], []
with torch.no_grad():
    for j, idx in enumerate(sample_idx):
        row = test_df.iloc[idx]
        prompt = build_prompt(row)
        inputs = tok(prompt, return_tensors="pt").to(llm.device)
        next_logits = llm(**inputs).logits[0, -1]
        pos = torch.logsumexp(next_logits[POS_IDS], dim=0).item()
        neg = torch.logsumexp(next_logits[NEG_IDS], dim=0).item()
        llm_preds_big.append(int(pos > neg))
        llm_scores_big.append(pos - neg)
        if (j + 1) % 50 == 0:
            print(f"  진행: {j+1}/{N_LLM_SAMPLES}")

evaluate("LLM (Qwen2.5-3B, 200 samples)", y_sub,
         np.array(llm_preds_big), np.array(llm_scores_big))


## 10. (추가 실험) 모델 간 성능 차이를 더 잘 보기 — small-data 세팅

전체 데이터(~15K)에서도 MAGIC은 모델 간 차이가 어느 정도 보이지만,
**학습 데이터를 500개로 줄이면** 각 모델의 차이가 한층 또렷해집니다.

아래 셀은 빠른 모델 4종(LR, DT, XGB, TabPFN)만 500개로 다시 학습해 비교합니다.


In [ ]:
# 학습 데이터 500개로 subsample
SUBSAMPLE_N = 500
sub = rng.choice(len(X_train_tree), SUBSAMPLE_N, replace=False)

X_train_lin_small  = X_train_lin[sub]
X_train_tree_small = X_train_tree[sub]
y_train_small      = y_train[sub]

# small-data 결과만 따로 모으기
small_results = {}
def eval_small(name, y_pred, y_score):
    a = accuracy_score(y_test, y_pred)
    f = f1_score(y_test, y_pred)
    u = roc_auc_score(y_test, y_score)
    small_results[name] = {"Accuracy": a, "F1": f, "AUC": u}
    print(f"[{name}] Acc={a:.4f}  F1={f:.4f}  AUC={u:.4f}")

# --- Logistic Regression ---
m = LogisticRegression(max_iter=1000, random_state=SEED)
m.fit(X_train_lin_small, y_train_small)
eval_small("LR (n=500)",  m.predict(X_test_lin),  m.predict_proba(X_test_lin)[:, 1])

# --- Decision Tree ---
m = DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, random_state=SEED)
m.fit(X_train_tree_small, y_train_small)
eval_small("DT (n=500)",  m.predict(X_test_tree), m.predict_proba(X_test_tree)[:, 1])

# --- XGBoost (early stopping 없이 단순화) ---
m = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    eval_metric="auc", tree_method="hist", random_state=SEED, n_jobs=-1,
)
m.fit(X_train_tree_small, y_train_small)
eval_small("XGB (n=500)", m.predict(X_test_tree), m.predict_proba(X_test_tree)[:, 1])

# --- TabPFN — 작은 데이터에서 특히 강력 ---
m = TabPFNClassifier(device=device.type)
m.fit(X_train_tree_small, y_train_small)
eval_small("TabPFN (n=500)", m.predict(X_test_tree), m.predict_proba(X_test_tree)[:, 1])

print()
print("=== small-data (n=500) 결과 ===")
pd.DataFrame(small_results).T.round(4).sort_values("AUC", ascending=False)


응용: `SUBSAMPLE_N` 을 100 / 500 / 2000 / 15000 으로 바꿔가며 같은 셀을 돌리면,
각 모델의 **데이터 효율(data efficiency) 곡선** 을 그릴 수 있습니다.
